# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Detecta qué es**: persona, animal de 4 patas, **cabeza/busto** (mandíbula, ojos y cuello: Hablar, Mirar, Asentir, Negar, Sorpresa) u **objeto** (Flotar, Girar, Rebotar, Bailar, Saludar).
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Textura rápida** (opción en Calidad, activa con Turbo): hornea a 1024 en lugar de 2048.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNrNfW1z2zaX6Hf9CiwznZKtzEh2kqa6VWeT2kmz10l8k7R99nE9HEqEJMYUyZKUbCWT/Vv7fX/ZPS8ACb7JSprdWU9i8wU4AA7OOw5Ay7JeJkWSTcQC/ti5I45+Fms/inxxcirsXzfxbuPHJ6dHx+K+OF5v6X0hb4tN5uPl00jGgczcweC3PBGBFJmcb7I8yScDIb4T75IgyUUE/9dJIPHvXxsZ+LGY+9nSx3cyFtDW84vfhI8XW/lB2KnMFmEkXj75x4uXr8V9gCTEk/N3r52h8PNQxIlY+TshcyjnI4Qik2IT+zQEsUMwebjchPBCutSNc2goldArkUuRQhex9Vi1/ebJS2HP/flKYv9TfxnGABUuz8N4c+uISO5gjAn1AkfgZ/NVuOWup37mRzCuiVBAM+iIGqso/CyArsjlBqrn3JOzSBSZP/Pfw2u5ThOoLrY+grofyO39fLUWdhDm8wQfQc8cNQBGEfU+KeRSinkSY1NzPw4Aj/9HT4Swf4Fy31NVKJMBZmRMXdddpZomisRSxoDIeZjE1WxUeMtkvokKmitofrnBQVFBxDijjbsqYDQwMfkKcRfp2ajQ9n6TFzJHTKWyCDMssg7ztSoocxHGeeHH8E8m7sCyrEGICCqww0BTGfTVXWyA8KAoTNB8od8v5+VVlMz0NfYkCsvbMNFX7/Mk1tdJrq8yqa/y1aYIo/JuVxbJNzPA/lzm5ZNilUk/CONl+SBcl3A+hClQsRzo+3izTnfY8zgdLLJkLS5enAv17sXaX8oh/3md5gMu4IoKA4twORj8evbmTEyh127qFys3CLPYX0tb3/uzHP/anocNe57jDGAqvfPXv/xfqFV21j1P5te2M/j9xdkfb+GFbUFrcWENhTXz59f4N5ILus/C5aqwnMFgcE+8BULOQqC2CXTLR6LJxTIEPoJZFYlY+FGOg0tECiwOJFFs1jPgBxkB3WTbMEiyAY3PBb72Xrx88vzMu3jxj7Nz7MKjB95oNML/A3z79sUpjvN49OAxNB3Ihcj9hfSoWTtcL50JkRXQyO9+BF0Cpo+TDMRW+MEnuuQO4ozhW7vw136cDJEx4fcCixaJ4yKNIZxwAdWBRHKmwDm1oWbDpd+qQfzJ/BD4ANrdyLMsSzLbAp5WMgFHD8gwerCl/gEGserNUKxgWADczcMPUrd9I34SJ8cCuYeu9rV1XkKWKKPWIA+BNUG4gVyFVu11GIfrBMGlt45qlpr4DoD/LLon4AsaXGYgeGTVwNq/tXF4DjSiZ7ACCyNW454n6c52zBdusQJCif0wsm1dc1jCcPQ0nD959cs/X78tG8SaKGd55mIg4jfPnyLJwp8nltPddryVWWGrIlQikyBQYnytCG0eST/epF6yKdJNkdvXUqbTxyOgnTD2FpmU3nI2HY8qCnwKQtYnCWdISkCT2IbyPQrNkLRViTrFK4UvSM6XRFhku6rTQQ59zoH1ZWAHSLAiwEGifHPxV8ny75Mwtlk8uK9/e+edvngDOPjOchxEki4V5iAq7MCpEI8jwSZI1oEgya+9TY7cVYfluFQQVP93352Myto3K5Av2EkgAmGD5rGDHGcekYV0TJV+MlFmTAiND9oOoHNJao+c2hvVo2wNSl3awVCES2Bt6Ukkx3z6LtvIegVjmPI2zGHKAtB/lgvi12q0ij9QNpPrZCvNYrVSX44aeTuXaSHO6A+o06r11AelwQT2F2jdwouSZW5XRPR6jpQj/G2Ys42EkgRE3hzEkQAToSQtkGUJyBoX2A8VBsBNkC0JThAuNjk0Cyzz6tlr1MKr3UkArOuQ8iewEmEtAU1ZItYbmaPplCdgFcALf4uyr5KKrHugp0vUcMajGz+L4VHOQk3duKBzCpnpW9viiQOWXIPGBPxNLfc7o/Pud1oq3glg7oPJk2S76TNS/39wCa6NvIFKkGWAGjDKAXXp5is/7XgkG8/ApsUnBsWogbtLWZzDpcxsbMdxc7iXWxnZusAfT968evHqudNmY9LiMC0wLzLLXSSovIFXNAgCuK5YH26wDQ9k1SzJw2LHtK+k5h001t0FMDnjHNXevl4UZi+Kv9WLwWAewV9xFgNwyS+R9j0P1FPheXYuo8UQbdnF9FUSAyFDc9M0C+PCmAAs5KbAilgOxYpiwEAWwBiGFqGCAAGKmmOgx8DsM3yBzdTfEBX0v/HW2z0vi70Ve16nPgyw/413HcYoGBX/lS/fJzNUBx8tf16g42FNBOgjS8bePIl8vPtUr+CBiwPa5wYqPQOrrGrrHhio6HWAgpJBiLrIxgswxJNtGCkxAUIBND5gGxy/QumqOnwJogr7Q4YUtP/Yxf4orxDuTx7Sgxm7JPDg0cht9RGAeCi1DXu2W5mpLrtotxuiukblZtfcTRqAwLCxPFCFH9hJChqq3qpj6MJ+Yq6xVdkGOz2AgE9NUvODXQvn9EqhwkuTJIIS84X7jqzxC7g/uwWXGQSjDTaUd5Nk18Ck0xMw2JgyLi31zLqqugzoWPvXMHVZ3kIYaUEvuVa6smI+GecgPQkRBpeBuH8bovifJ0G4TNDJAzrboC2tfEXDe9/5pecOYjlZpxHSArwpwflFFs5gOOAJbCR6yURTMbq74GWSDsokSIF56JMLmEtUWaXS0cYBz5YXhHMQF9UbFEkwFh+UoW0pTgQpjgzl7C2FPHlHwYoHUS9YfcWQG6HAPm7sq6r5EqoTjfSVA5xSC5/HX72tapoHmJ/DaX3wiPyxf70tAgmkyR3YLo7DVgkw5QJUt7Zl8osFpmxOJRs25GW92NXfYawBi8ejv/HDnFGyG8w1Ch/vRqIPnSuVl0lA8BSGL8HUiov7RoDNojluPV5vYSpq7HoOZmdRBtbKcBQGryjGRRwXyPd+FpUBtjdPXtZ4jMyC1YaU/8KfS2+1mWmbIJ/7QMw4y8B/WVUFbH7UQ5dX/TIYrbEMTbE6DNtxaeBtcxymPKN3XkgeTk+xErbcUikXGQkFR0/Zqr/fQ4fByrUXLsVESPYTqAU3t6Xn+VWvNkDTSHZoGrA0bOsVyct0A4Z5JCUJtDKS+OuzCUxpw1tR/uZoULEF0EoW2NSxejMxIHtUe3ITgr4kbYbFMT4zgyZmm8VCgu20nI4c7O2ijRN22FAhdONrBk2xArMfPRA//SSOG26ZMV/ocM/60a5GGHcWiHE+0F+cORUG0CTCwJmLvwyzjkbbzdHjRzRSeVvvR5EUPurXfLO25S2oyNTOwIWkGXba9qK9sC6YY1T0VgWkgVkm4iMDU/7dxB0vPonnT/G11R7awvpojEAciWLijhafckOLKLwQVEMps3pKw1QaEmKIkauhgAqzxGnY8k0nRnNtKTJOw3fPouTmpV+AYIiXFwA6Cg2DE9sCHN1V3sXWUGOB8wC3gd2gY9XLRRKBBJ5Sf8HL8TBQB1IsbzvrQDo0oPqU3RPPwFVY/X4KKA8k2iCLcE5LAbwK8R/j0S2FUzI/DVUUvlf86OG5MvZBN3gLBL0N1vZ67vnRMpla63nD21fs/m6XcsSrDbCzld6W2mxzgEBpCxZRogVtpyDM0yQOoaEOkXIg/M+GrQgWR1nRK9nTTKg34B6AXUWzrO7IhDIf0ISbD/5Cw7LYkddX12q/IAOaxqYr3mFtYT9UYY7vy347GIwEkxXjubTsAF0Dl9AgLVp/AOtJ5H5ECxTlMoz4BVCEiykcwhSFv56FUIAWprR1C8BLUGo1hwPM0JsRd0d7S6oXv1CgN6gpWb0ogEGPStpRCAQD7WC75O58E/ggqIr1JnKB4JMbr1icHAN/ItL2VIrjzy0+A9tiBY7ddbN0Taywv6x6/hQgLLMEEPeGYmaZyczmfHZbaLUS09Lsgn57Sewt0w3YbBi+SQqDWu7WC9C24dp3tlxz/VujqMOqmqZwZs3J7wdOnEQ0C1QlzHVSRbVMfiS1fMd13Ybc6ZQqjQADO0GGhug2G3tVcd+PtVLVg6MgLI62x0ejI0IACIB2gPUL5Auj4C7B1UEiNbps0lh9dr5obnhWMKJI3NoxL7XQ0IET0IXQto+nh7Pe0kj6yY7iR18wOgCsaG/rZyGtQOSFn38B8akI1kHDB//k8ymwkwjX269LhRVC9lFiY1rsGsmBdOqZGKc9a184Z/XZ2kebd0/bf++kdU1Zm87/xmzV5mmIrtUmRyVLKxRjShTomT0OrZZTwred81HTdhz1b5nQF1i9ZThTjzcUi7UvDVxQY6YUvWqoFQn4AbfZAQumu5511fKic/THaQU734O3RRdRUe+F/fFbav1b7Ex5SRC5Q9+yRfPtpw5K22v39r5oBbq7Ebrfs2gRXbfIrbke3b7q3aa9QUUWT53C0GRvl2iV/KtioHOQzmA/eL1eAP3tdurBbb7+Wu7IIcp8/0KHabqV/Im2X5tHq1UCXILiwCe4luEHdpLRiISHZkC81cCllSwW6K1YV5M7Oqp9ODL0vXm68VTVhjOHXHOH3aMS4148qeNL2HOQYBkuZO1wTTdIjBwpx6qJM8RIl6XcDrSltOZZk//DproyHzid4bfUnAaSoOUT0xztcYHdIrEtQJnldK+F1Aw6enPjZ+tNandGY16qZYYIVCGHYu4Mq+ja3lqi8106qqqZ9mLHb7GPDpwfYPpZHIKNDl4d5+5NaEkjjNgNvZZZDFOOM/XLb6dPxA4IIFxKY304FxhUyMJizeHX+eb01SvOUQQYaQa9znQenI9pfj7GbTHJMUrmh3mLKlhTcwxqRolJOlRYKT3tWLErSF5mmHv+1ofhAUXaDWJgZ/9O16sd8O1KZDvN/Js6268BHufxxPJGJeAMwZceHw8F/HLgejQU6p/jtHJFSrAu/rLDdaOIK6MoTHNp2+MfHg3FAwBzcgIXx6ORQ/G/aGrD9VCMf8BfD+HX8cOHzYaAajHhYhkhnB+h+vhHBDRGiD9UgB7BQ8wIGj/qBIMuJ2P9OSVWYrwykNtwLqcWzoPlgL8fg4EJwg1EfyPQSkFPrh7GC5mhXiDJZEP7/NzfFMncz3EpBsENRVCAipvySxRcxfiR0x0YsymXaxquh5iO6FUt5IVM8ynMRjLHxBsvk2BzbVA/TMfHj3tsRQQxB3vmOp9qicW31tVQZZVCp6ZLgErZVB71E5eH1jJfNY3bilDzXTxfZSA6P0jbGfQYPhjCAVMZlCay8iGi4rMi+0IDX2OSbCKSdViEgTY/SzGDuskrEpWaWKY1rFOSvDKYjurSBzVEgonTKqmOMqZ5EkK/zLc+Dd8NQbjPow00SYuwhl1AKcH881ay5DLFDWZQ4yorG8+uOIuLEFfJxFq+T3BtZhlRviYGe5O6AFrApBV+UWRqIOYi3WHGNFTAcTUMahhOtzHd6xpBDWEXhCudf9jnq0OTlZ1ltGQDrsmmOlLv7ldljrZjd3x0qhNz4CovgLVlYDTAs4iyF/DBN46b49Km7VxOTkajq0aKoe6NKozZZSAxNlEkZkmwA78Ok60zGQCBRGB2iFkZlrIUrYAtYuOF0wrCUn6iGYP1aI01X+pFgyQpptZ9oJQCrch3+PLtc6tBe/xU2L8/eftO3Bd0L568cCYqVgpEQomJixDUMOoneJLDQ/a87lZWHQRkrgT323tqlPUYZlPV4FIhDFMnfOY7XsvusMh2Ou0xzjHfc8T4cQ4pWFsgx1owiHwOoyhyy3HatK+mwU0V1eXllZ4hjSqF/U5GIFCzLASbxQi+wv2bl0+f19VDM11G9bHyJfSCcyevYB+8TIVFFfyWM1KDf4MptVDqaOw+AAywvcdaTG79yO5rhWy/2ojvbqckW2qmNV1a3dUVXY9E0cROpqSw17jJBQm8pPSS+i3nTlo8VHPokbf8nna/OkMdlZjvXzrm7KJSGCijsCYNKHudNDoLFlDGoEgwubOWF80cLNe8TWE/Z6uHrLnrKwcknj21M0KXhGlO/Uyl7NdXdHWL7it/LYN3vAsm2z2DR3a+WSzC26nlpvESDRsZyUJOOcBES9sNdqdUen8r7cXKpezMRuoWpZepd/2SBUx+yjI0uqwW2GdLjzy3aZy6fpb5O/tyjLk3+tcVGGXITl4si2mDxYxk6zAGT67RaJWITEv/gy80Hzt0xh4TstGFUmthp5VtCMhoGG9dliI2izn/isqOwQjeG1DUBOjlaFZNiXlt/ZAUh77+WTzk+NQPLngCMLuYZJZfGupWr6QzMbrv+K+dQBkXzGKwMW2YLmriBF0LnLscVWO43CQYUMR5TC7HV46ZHdcZaQD+gUtclMc8o+aqZSua0afZjK7TJq2pQPypG+3OgfXMT8y4RrfDVQ9fuphhjLNThBTHYCgycxEt6k7z5Ut49IaeDHqCISorzEQFkoner9JfgxvSFbK9jZR4qPpjq9Qw0/XgYkNhdmaKvzqkPk4eRmbNCXP+dlbPHFeHeRcTbyJC1aEy8Zq+gI5CNGIObUNGicic9pl1vKnLXvzZogPNFdxtmGF6JjaVZDu7z/nkhWUq4+FKMSbRB+AZqYQWSsfoiw0wA476XC/c7LehCMrH7doFEzqopcncx6eN7JkRZ8/81392Zc9wBg06dB+XCoLiiG+3mb/2lrNvr7C69Vk5u18jpU9vdDOz+lJTy7II92ZLTnapyQZ0C/YIhbqVgOUb+8/A2XBa6q4MpFAaGO6Fa7oh9Z1yg466r9Mc98osPNoXAIZDo+jKB2qI0pWvNk7RPqvpVO2tohhhqQ+x4iW4ZkNxcuWuwZBzxE+gDkbmwEokcdWkqFroGl+1eG83t22ZNvj+jV08+ZmcA0MB/wItJrP3kvYHoyOWIU/zDtqNH2S+crPX4IfKSlwjAjpHWpbYgfK7zblYnMQfZJbYPuiw8agW1MWUt9u8MZu3YEPcjodiB393Y4BxmzMGh3QF2hWuduWznXrWgTFCQpakto0wdwQX/M8xwYW/pnHQsQeR2FsppWoj33dg4IwfG5HWv7qDeGhk9gbw8r9A4oORwBRt27k4EjcghEAwYFW4W/Fd2+eFqqAKUBPb49HxA7C34HdrPyApYUL48QMWW/lfhnWcRmHh5SspC63P8Rr8vAz1OW98HeJu1yG640O137XmNp+GW/CHaVPnKnlPvgMv1fIG8FhiPCVH25HyBzEdaZHEQSu2Qi3vFQjGhoJsOfPrrNDHuY3dc9RGaz+rGjVKFEaSIUPoHVhafXxWYwLsV8kFhPay3Jy2NviuH+/MoGYulzkGs3JAIvbAzwpKIlbrxkNDzeCiRjgUlOIrwepEExS33UVOKxKwLUUJQ+5Y2mi0yEloYWNFR6GKwe0H1drZYY7Q9VPAZQAEjs1BO06N/7sgd1VDOYHjNWr/QUglNw9M1hpaEY85rxZTRAQfIaFBOWAstJlhhv4ARh65owdXTXmExUlYT0hUArGI1QbkZT7BzDbc1A32MhjreSspGWsTA7nEX7Y1bMZGdO/IPQmhB3+AKRA7QxIx8ABlUvmU87FDysfGnYF27FSdRc+NQCU6+EalEyzd7ENVK9kU9Q06epsgCB0Wuw4C+ADwqIEh9bjpHm2KS3x7VeauofJHDjCErZLh9BSFFbkVrZwIxeAA8auYJRTEqFwXdFxDHXNDo3+I+7E9zOvPD00ftk+lcvrkM6inMuVwIHgTbOZwc2DGyDP0vGSmQFRIwJ4hE9Ve246Nz1ulOrvTKtxrW++wHFiztZdrJKTylYu+x1tZNDTqGpgyCDxyKetlqW0XZSQ44zlj2mUkt0DguzBeesr0CTZApGiBe7q6vacKzqYH1oQHDkeIWQ6ehDnbU2UegSbwVgkwKypwukBX6Xg0Gu1px0syXNxgQvHmyYqcyKjpU6DvscYsUjqDgzHjdPotjfCQMZ1NZ31NeJS3MER4RZbOmnBpPFAhpWmDlz5zGaecw56Qm+pZR8CtM89lz1D1cAHDYEX2zTRDZmzwVLuL8NbjZJy8Qd+9FaJIT3erfCd+jvcnLlDoi4+A4YUudm6Pa0qslCmkLFEJGBwgfq7eT7oowBAkio0rKQW6flpJrKbUJEQ3dkwoYYe76fIy1slrl82Yp9pHgad/gMNbd9E4LIyujYqgsnfzZasYnfSiRm8u4KtYLfX9Up29ctUXsG0BvKfj6mVqewLGoGBWpim8LzHI5k9oKydG3n3MayjX1gY9O3Z4Stomzmfn8un+LTDscGdoW1v7FYJQT9mfMbNOMxpOAD6bQFoefFcwZr2JClxv/Hg9AaMRrYprZbBSS25YyDVwPJuoBqVUlohONmUziuAh84wrOxookHMM9+TywIOrckeNSjDcnwl05Yg/m1a07otKUGx0gbbOGXRrgtzTeK3l4qqZyhLL28JOqwQmbquZiFTRB24qInRxdsudGU+GyMopi7mrXrGv2oJqNjQKEhDqQfDFKeSO96CjHjts3usn4qcp+ESMk4dfltaVFCuZNTpcx2mj8+2xdCZ6KbiNZK/aU0RVd6IXFetI9ir3QVXrgn1ZHXKdFjveE2o3ozjMVtBJNduEv7qA/J9ZIPkbOTZK0tfSbPasnigK6rWp24k4WIufOsO9uTeDL1qFcerLPv0d68zl6VigYYXCM/pV3B4Vd68l3+gTDbTjowQ+mpb7lm4sSy224Hl0iGGyfULM0IgxxBOJVZKB5gwSl0NL0/LUQ97S04ruHLQQ1KlkP3tRCJqzmwN0xL9Mm3YLLolos6XDQG4vebWAmmxKXnidJ68wo50fXV+xNuRzeHrOkSO1SHoSd/lfO3t2kt8Tv/O2iCo9Khf2IqMTC7/Hwx/9CGbBdR3B6ZZgbuOGcBElwg/AhpVNX7704n2Kc9JElHya66gIXpM+ZhmEt0DaX+x9aKLB4HJtnwcd8BSwrT3kDQ48uA5H5M7O13pYxRp4H83XYb1E7b1FvruWu7p5laSFGWhAMaiOX0StNDbEfUkjdPhgHW+o5E3qaMf89iaMr/TBL9cukAtJc6e/0BY01mxXoHt2OZmIH38wMLiqHSETbNZpbuMQh3QkmwfDV3ubOxpS2F+5K3kbhOC1F9jA+PiqMlKzTazQx+FnygTT5yABw6hL4hl1TTFavm7He729wezmGtWwbdeoRli3qBtyxdQ1ELHuB6apjY9PHtShaBUypYNJoN2ptdrgamkY4EkhK8pImo7dx5iJBSJKTkdQsD75UDSZWr/4YGH5mTXkB6j+dNOcCKC7BEgucLGyY0xKpKltx5mMQrnFFh+bhj6FF5YZxhcifz0LfIEqZCIUKmIPY6ChH3UgnXqmwhJUFPh5qfEWgAiOdLfYv5zqPT8W6yVWICqsMWkei8SH9FQEZZ4X12lQUqp9pwSq7WC4KEXIoGnUluYIqp2LS4uua8cNEWWokuqGi/KNWZYjBFyUr7kkXddSG2i9HovhFRdq7OooaUAVrO65eHlvVlKEosel7tTI+K7WjVEj193MACUCsYFYj0GJ/b9NWFCSKi3vWGY3QWjxiVikFydGuFixeMmDjj5l1SxE7GqWaRtfrErNSsTodcDM8WYhemKW+tS5JNUtVU33ll0MczlNLSlVEsjR3m+n+2FK9m4TqYZONChAHQyaEErbo8fMah1g+oxSWvV5xJgBSGrWmD4U7XgeZThXbDCthQumOmrAIpF+s0yE/83M3W6zuZSPVcgBhST8L4Uj/ynlI/8Z9uAHRSX9NsVkeVXJSvW3G0wlQmdJEtnlrVPm26gXfONUopT9BnXn9EIHSashzxJnqOUmPeNrp5KX9FTdOFpw0u8e+JUg1b1U94YyvtYyDy0W01Yxyeq6JG8+uoe84trBqdVLoMpLCx6BAOmz+d6UR3XKmA8DQovuujOXswa4EjzNM2idxrrW3lO8SJTlRbYgcWZ98+/frL8JvG9+/eblN289C/egXnefI0cGY+PwuMaprHR6b611PJEaTz/Li00QJu6qWEeWQ/s9jMqoHrHbSlOiLLaV0uU1uWDSCPrgyVzNgV7wgV0eoX/YNwos0jsMWgXsitdNPtPW3HLWaT2DmVpeWGTfex/jT5TE6hgK5UyTozROVLwn3mDwnWRTBL7mSHzDh5WgJ4MbOfAYeUxEQxPFz6U+JR/P3c7CJDMO/UDaOLtUx8ahb2afXZbHxnHEjnlZ5XchMZyVp6hRjQcVrjx9CgKuHsOrkfvjCE+wrxq4j20aFVij64pljTt7wXBqJ2IV4fzaBsdohgJ2jottOWVEJWljrsANf7L14w8k4meYpCDyDczNmhzFAFN9SOrT9wEYfdmy7r0T8sY9O96q07PI4IIOYK5cjsuGbbK4VRFCGPjDIfjo9Q1JYxwpJc3SiMYd7klpcvg4ezOo5eM6+S2Nv2MDA3bnxg8LaPGhU8dgCh6YbKNwETclF8ConQ5/tgWkNTBAedRVET6Ryy4wW6qY4lwNBVzn054pAw73wQKJO45vBu6h3Af7gLU3JTUXcaNwZ351iZ+8tcSr2iWuNbBWHrZ6WZ7meIVB8LGpLfSB+iDs59eYUNnIPDRsxhMQjGcx5sX4E/2RC+BZdHBwzSZRJwtZjcCkbqFDwTQ7d2R2rlFKH0/ZGEIvcuOtOjitJvpIZBrycouWVd7lWZMN1cdBOj5K4cBAxfOJQAFPwCpKZgwN6XJotoHFUWC0y8tDn05OLeQfq3kSikWO07YK71g9m+zbPwTuv/6TDxSxGqsfAObQ7rLem5jHdOxb6/qSk1UaRmdj/bNj5kpJ3xBYeb98KifMOg/XaUjox8zKjGfCX27eyyzJra71S5CDfLZbeYCsjHnjmRGKtTHihGnXyiZUG+ZfPOmCeEF7HylrO6MlZrQvcm6FdmaPQSxHUUL7Lec+fjWGPs4y61gNvSf+4+FIUHIfr4pnuPcED/DP0w1l+qnn1Rl1O7gOl0CEbkfq2CLxaGzT5op5l5NUFf9ZnIzw5wuO4Nib52Msw/cCOHDFnju49+yNQ06ZaOciQJNEA3Rcrl/LSHD2LrDrTCjua92kvBN1NDXyFrFUN+zIMLZoqmlu3GU0s5yvdtAf0xBN+v6R3hPHjkHimiCREHNF1Ske642BciJQHXzGbyWBJQmsljh3sjTZbSMSyhbNPu4eJhPqY0mdk+EnaPYjJ3vANTVudVJ0O02kI0Xkq1DugVTbw3y9s89mveVnc5AgvXPf6yB1VW/4RzU/yWZCs7iMOfkgB1NfjCc1uYPLpPLzdXFlhE/6vGt8GUAvrpVZz+q6RiTjIRv9pLZLC/8zdZZ1EcaFVt/mcsmLJ597RlVNt+rFweay4N9RqfqHVgQZhyrowHYASZwOwdB5cEwPnvvScFhTc5k79bSGu4+i2fpUC4DhB1zq/ELi7oO0n841wdQpXdP58YSIgHZfHi7DezdQHZyPcNgGo9pGo8NckS7LvGa/G441+twP67729+qRMtYGbU7mwBWRUb35mvttgK4eVsDh2cAITeivyi02MuNlcfXBOTSwyKEBGcRZA/VvytHBRq4B6ZSOWcaEziQvP/U3AbccjDyiUb88yXyIZh85SLwKjR/14nhhBbCY9R0OGqfNwNEdVGp08qI6Srb8Ko46/pVSA3jmJpQrzz0ScajX7/GcoN3czJcnQqdlc4vHiEvhVI/jIAoEJ1DpIle93/IwA6eNQtpIVoV0HLVaocgxnMxLFPiJAvp+APjncG1hpwIQXHjNnSuPWKD+VysGsgXlUz3mINGYsFW0ok6EXBfomWgQoyDGk6G4dQY9RglKPHC+Ru6PP4Cso3sMidRqU3Ck1pNtf0+2rZ5s9/REv0NkUfl8aTLIOcfogAso5EjHroCcBKuVvos3QQpSwVH+Lp7wZ1mYVYRlwJKx/uaKjhJEvmYOtIfdrxOOLQ+/t2tWyR4OAfwXG/DU2kcyNKOdHGEt1/eboaZs3s2bi7s+6QU1O8JtJhqgRHOc5cFZ5mci3HwzAyvbbodNVCnco5IOOX3CWKDRKzNqGcNYeCmXWxqr0Ho9pVw3wYnRYoA4pV5BWy+20o+lza40pLkqzbK+vEUXxdwztGh9ughPOFTMXTJ2M+EGv/apRNKr17RM5tO3I+gDq0u0q0C8y9g4TRtPsonR8W/YKveEv8Qq9O2yf764ILoOSrqO637yYnvwRP13TVbvhDHOhixPmi4lfxOT8MCaDXWAVpmVjnv99N+G4u2786FIQ/kBj3jl73TsxPnr02rObv83kmtBBhqnNtQHP+MPbi2kyx+n23dKHJX9fiqsP+M/YwzNLW7btT4rs0pjmXrY8JcLbxa1jeSZaWhVSx6GPWSug2h7aBb1j6rMDrLPhqLjG1OAwBvL+fw96zpuVN/BFrbsGo5LwKRjfmnS/D5WW4iGzbUT1UjjQ1lQzICS3u5v1qMZOKTx22brt11t3zb8Dxqa+eEgbSXyF4h0plZ4yx8Oqr912ukMHmu+KAk8jlrg4UnEllZ33gLXCG97HA7o3uW1h3QU3uLFHZ2nDzxCm5dXDmCmwL2fLbi+R131VcNqYEZN7Ba8VVYaNdIq4/QcNoMYrlGZxiBhg6I7UzwsIspth6GWUR+0JChRm3T+VEXydbaA74X+lN2PYfm1BY/clymdcmYXOmY87omWl7UUxLIewYRaVQnFq2WJWXQIWDyTQlepS4cRw29nLXRCVN+ynnJSIH+ufHpxafEVnX6YbvABpfhX+K4kBiJVCY0wZUGBkhzXK/EURs4C8/x8HoY60UwR0RQ3cRu5K7gXt82eCj8oLBEywndvshDmGWWx05Xj9ONjXK9apz6YHRyQAc1t8PQHEkCbovxsasMHwc201PiHtAM+nZhkneNpXLWP4pSZN0GYTbl6mE4/pEMizinj6Vruptdah/E3wKeLrX5AFvqUDfWGGwDueJivPLZqbKjpHLBla6s026ZoK6lKLm7vkIvUZlMuGox3h+DdJ0m3To8w2nYB3TpOn+g6RLBu7xasvRAPkXbdUm97gNS7S+gZ3bq0SiHAtECaX7ei3/UsBprCc9suUmPqr8HTBnNckvTlzsaL9tczyu+BHSACQH53HaFWaseVEhClkbatSYoDmP2LdgvyfHQsefSG0rRLTkGLq+Yh0w3nUH1Imv3Mrm9JmzbLYk/az2JL+8KxUbAcoghzNu2afDEsvVbA99De1PCjM8PoOK2878jrRnYElqRn/4poCudrWaySoNo+a86ccUjVGqU7iHgU7nBj1XMSoBId2ffPMKVz+qAEEfiwevPiwjs9e3b+5N3ZKX8aFMV+hB9Kno6J8D60k0spDKE+cw4UjGc1Yr4debPteUeg2EmjxbfvXr85O6Wpc/FTR9hZ27bcdfoADTt1jKClAxzsNOHH0f/sXrXESFzXiFqlPyhmafPdAs+JrjDA+6TwrpavlklwnecKkchDapPGr2FBZ4lHeJqY6U4e/ay+z42hMBcwSivZga9dT6r/8sX5GaDk1dlb9B2qzU4q7DARl9XyEuVdaD9qok4/1OswXKjb+qlHJlw8wrsCQ5YRQaHAly7XBwk6f7srv1xJ4B6b4IDR5puI4HGJXkDkUWf6FB6EdFIfHxXgg+V7oaz9lPMFlAuMYPAs8QqMftGLGg7XBTq5x/PxiPmAMPUQO3TGBXBEz8+fCv36YHg3cqao2cXjxy0EQskHCX+PDN4fDgyMFXcxuyVgjc49e/qPA+Aozwnq/4BkcEG3dATSagMIPwSCnvgfcOJf4tJPSQ4HjwO8OQTx+BGA+J33gclc5WTkwj5/feocDEzJCHXGp/WE7R6DFS3HWOJUAaIOzvodX9T4SuZzGfdwVScFHxtwSl5iGu6DotYiiKwwZkyMgON4o5dSsCfUaTruuSy3H94yBPo6eTSiiWqBK9/2AOEea4z+ziFGzYoKmZ+q/VHKcNHH7H1peMsIaXWYVnpT0KMhR0OnuCvb4hir3hhU7gWiLTbNLaTzNX6Z4FLlXz89P3t1eoZfUT4iVXN0tPDnRZLtjii9cZPSQ1oBGtHlRXt1uyuUr4OByKrpjgy5oyNuIMTZJfTADWCI48QtqPASKlsKf3DHKLToODCbbxgs41S94BunC1wZFGXc6ye0EYcrl/cMWG+74ZfqrhN0OWc4xLHOs+NHKteO0XdUiWjqbd9mBCipVsMqiM1tslUBMsKqdjqAEbFYimioI9VHvRV4TnL/l2mpd8vdvOVinm6hChfR2URUqFLi5J2ohhqvquXCyqyg88frH2hOM/4+kDpKx70gZwEoFwcQALVMzZcvLs7oOVik5vO3705f//burrwJjNmoXW+zzYKOHxobH//GXNJp46w37DBagGnmcm9aafjWEzofTSwyfy1p6YLqqLqcOcxGn/UM48AdVuOarGfwS9GDtbMaRGH/GXzvwMwhtE7fe12utCihwN8Y4x0N01L8d/uNVS0bs8IpKdw9AROb9vC6ePh/ao8dTAt/QB8q6ZDSHT4pHWUcN1wePnxeHSdHyMnUeWnttTUl0Ph7d/0mhonufttBl+pAvW7IrucxVSuiO/+ORdEmpJ5vSrWI4RIlntOcu3YP1YmD9rXcDXWWff30wXXeF2wB50X8PGXKxoYAhsZFf6Kj4gM6Aa//hAWDcJqp/8ArlPBvsH2yLG5x6wc47hbrDiKGWjQ+xU+moyTF/coom0Y9KysM7fIIkzsn7V3l/HowGHhnr56/eHWmJ2QwQM0N8sqT8RK/vgEFp2kGhK7Qt4ySmR8JVW2guqWhtPbyVeDPKnhlPEJ1RgP7/zVx12A=", "blender_rig.py": "...", "app.py": "eNrVfV1vG8mW2Lt+RaWNu2xqyBZJSbZEm15obI/tYDyj2J57g3CE3mJ3kWyr2d3TH5RkQkBedoEFdnGBvCywSJC9QJBgsRhgg31IkH0b/5P5A8lPyDmnqvqLTYryzN0gc3Hl/qg+VXXqfNepQ8MwXgepiKf8I7sSE2a+jLnrhW3mCp+JJM3gZgg3QRqH9CwIUzEJw0sWMidchCziMy/ge1E28T2Hw7MA/5/GPBEBZ5zB1VIk8CnLApZmAYCA62d+mLlTn8fCMgxjz1tEYZyymR9O9PWHJAz0dZjoq5gHbrjI74S+SoQTizRvlmSTKA4dkeRP0nksYGLBLH/gLcRe3jFNmvEErvb2pnG4YBZT72A2U2+mHooAZiv0q5lIbflkb+/sm9dv3rERGxvP+AIwEhsdZjwL41jQ1TvuZy7Xl6m8+pJ7vrx6K6IwCfHqDQ/CBC9e8Qm+vNg7f/vi3Yv3CHu1x+C/B7AGi0kshsyMeBImHRY6aSxEhzkc0N5hqbhOs5i3LdbvMWoCi7UQABdgLEJYxZC9z+JJSOAMujQA2nGHHZ4cddjDHvzXYYPe0Um7I9s8Fyn3fe5Su5MOO+4POqx/vN7wLY882WoA4AbHDzvsaL3VGWAA2xz1VJcn623e8GtvQa2Oe7UOj3qnD6HZ7d6Lf3v25vzrF+/s56/fwtyMAyQ9oNWDV1lwk/Hg8Hl3cMCTBEjjQFzzReQLG4DORGLs7e25YsrsqeeLxHTbQ+o1jF0R0ypKPNk8gPZuaM38CS6KegqcUnsSezNrOrkunlgTXwRuqYVcFe9jDo06zP/TzYDneOo5oRVOPpS+zh8vUr/0GAgxFokXBoeuleCbGtDIEx95oscaCaAFfcNjx4Ml1bcwAdsLpqGFjAdUh59PkWDG8EnE07n1IfQC0+2waZtNw5hNmRcofHlTphuJay9JE3P9m3YO8guAOS1AJMBJwjWR9y38s/atsW+02+3yxKC/KQoi/BwAgkyguymwp5tceencNA3LD2dGR3cLwiGLsbFe9iRbLHh8Y+KU1dpzmCxhANjaNPiHLEkFcuLqVnbuA58TQvKhmFOgUsAhSjT4+/psyPb3Vwhj3CJetJHVWxfDzq3kzf199vNf/AcYjZs5novi0cDJFL0WXxltJvxEMAMuvsh7nBqqA2ouO2l1WOtPW+1yJz/9Dy0E2IrLpnDfat+y6BpfGiWAIvkhE75IQ1YCDNSArc3So3mG1AOd9eCFvOkwoigGwrACsvRVpD5qdQFeG/suvUzELAN5nthpCOJFzeOWJSUynhpfhfGCD9mqtcj81FsCefFWBWnlFwpprT7QVxq2bivDMpu6niJ46pVG916hDdeyMqVW4tW6VRi2vaLbALts7EY1LjoqA38LdCviYWUB6JFGXAUhP//+v7P3ngAVlAwZDZ9tnliB08pkmElSv12fEj7MZwNzKRPSllnlvdRn9qVP82j8eOLnc2z+eGW2lp4rgMBa7AuWpLHZAEW2aLWBS6BZwkwBIkW+ZJEPypSwDXOqTHTt83zKRoXXiDB3Idn9fbVGUt5UmJp0iANCGsSJkjW5PLF4FAEaTOOsaASUB5NBoSxlIAmUCpQLJQ+dishyfJ54U7DCsJXR1gNxii5TLyIjYGXMswXgBiYPChb0QgxCn2OPTsbdOIsE6XDZJxlsR2BIpDypKxhSMg6fgJbB9vKKHbAJCE8yZ0CLgWjBd+rqNgcAompGoxlfkja4RHFuosqKA04m0CTmH6Ux5ICI8gmgHAYtp0PTvlQifg2lU7RahJOCwpWmKohOQgB95iiKhwetdodVbtu3sJzrE8X/vkChz8wlWsUrXH65RHIuIOFInquZafndXicJP3RtEtabCeLrb5/Dk5mIwzWCqAwN+OQShc4yVwAwCMJnhy0RpZJ8ih4vLC8Vi8RsNw1swcFgtGPhe2IpNg/urWwwZNge6SNAaQOWFHPQzkZXIZFKUYkIo6GvpYjBpBEoRny+sS/QAJ9+TNC3AIDYMlGCctyqAJBqdvnp7+Uzav3pfwXw4YEDgzwAep+Bgb9hXQ3fmwjwAsJEKnNQapIO2Q2TZAgMAFIQdCXMFSQMLXImgqaJSe1417ReUSs9q3xSpY9bF7CwiMPSPLDrCQfB5gjpbQFQsOR4u2kgpH1xNUHU4ty2jOY81+QO9+FvhPYh9/Nh1UFJdEdZkOIcvOmnHx20pDfi10w9ck5mPEYSZWkI9hcMH/xBBI1oSLKELYXjoQO0cTIeGfgoCvpt9pT1d54Q2INyfQV1quBUp6ceEt7Bgby+YYtPfwCaFjF+WkJxQbuJF9j4MaAGCQSRrJlu+9CG4J312G+U7bggOzLxgKQYR89bG2Ub8KmtmRITrI8E56Ffs/ypngbKiCs9VqWpwIoigTu+2EwpP//t3/3v//l7trq6VYCUcW18H3wfKAlFn7TB2H77+iV5rsZZJtUAz6RieEUKCJ8Uqgg95tjjJC+GFV1UqB1wrys6Jtc6+PW3dW2D7jbMMEcnvgq8AHQ4aiLpCyw9cSVie54ufNMdfQMKtsPmwpvN09FD9DbxvXp85cVi9BUHud5h4BFHXD1fJDO6UkgzDOOFDJ2ww+fMxNiDAL+qjaSXAb6nMV8AuYsYrI+SqATxHAFUCrOAVwdWLfqKGB9RVIdOjgs+F13c4XEZKnhj4bxABxXLqZYLVvKJ6y3BrrrxxaglZzxcyX9vo+vHrpeAAXUznPri+jEHVRR0SXEMHYERo8frlGmgz+RNb7rKDdctEULXBdw5KVgmQyf0s0XweMajYb8HHU24czmLQ7Crhg/6g/5h/6QJ9oR8zS5GarJk2D+CDwFQGA8fnPJTl/PHU+i0O+ULz78ZJjfgvC26mdd62gCqPG/6KvE+iuHRw+i69fT//Of/9tdPDqDBU2z1FBWvCcuLWDfeZRMMY3FyLkAxRJmfcNbUwUSDV0MUJ+JUiNbTlwKkNY+RLFB2vz4D8C80cT45mDxFbw++lwOgv5pbMf6UxT7oKVjoOMmDUD9kYSqoyQ/AaFMjHYFgSE2Mb1n4x2yzfRQ14LblhIQkXRDED+iST40/WY5W+KJohuRea2b8CfFAP29DbLDWSDIHIo+GZ9K9nEgSOzTOAxl0s3nkHWAQZrSqk3CrTMJgZf/p6gc1thIJa2aKnVFrBX9vWxr1V56bzoHCer95vE7ckpqGvRqlNVHZOnm2QET74RUQT+b7CVg8InjMruNuAsMHndlNY/gEpEyrBlx+VnwEKyyHXyPSKmeiBdUl/hvGOPzHoDhnXtBNw2h4hCT7hDOQMNN8/im8F+moBR4WDy5bNQI1qpR5MjmZTKePCzboHyLMs0nsxSivIh5Q6A9N6MhH8fTzX/zNkwNeUOfe3gPW/QX/sZliCtTVRCgJQHzjJWCykxMLvIZmF4wBdBALwcPxpFOSR/nAD84C6RuBDsVxopp3uIu2CoEzpYLtPs39WbjM1ULbYs+gIfG2K5bwcInmSCGFgdnhaxXrfgC2BFuAmQbrjMoaRBxFfAEelxOwGGodeB0kCw/Dc4jJpbcMWeBBc2RcCgQCfGvvX3/7JalIrY/sK+6lZpJ2WACk0WFgiMygYVJWLuCxYfQI0QLKAjqa8A8hA9uM7AccK42D/RmC+DO2CHGs6BJDa75E4zHXK1dzYD7qSofUknRs4D24CnlsDZ+5gF/jouB1PTATZm/2rNPTDjWbgnvaQa3mjMA3gSetBRoiGPopC6UuNm6lPTAkrd4UIxTtkp8K7RJfiAjgHudWF0IHU6w8hph7IApnsfUijsMYnQXkMbBaETi0hZ6JRGXQLwpNJOMCk8k8vLL1POwQo5GwtomHWx8cbSdaPCA1uS2CC8/dBajBQi1LKewFsCZOKoPIlxiLroNWZiE1s8DQC9DOESYM/cssTcPAcnzPuWyjcAfcA00lZVG3Woc3ZGOcDKAW46+X0t0sqAhXSxo1aroP2HdqfmjlkQ4ikiRyWYqPQyQOR/iZz2Pl4wB9uWiVBmwhFkCzvDykst2EBpDxgkhOcYMLTPA+RIpL0k9/QPr/6R8OkQi0ZaQi6z/98xCwmrAJqNRgs/+gZNeZDFoDO56H/qcfZyFuaihWDHI7ulhxZRxRBFpiIYK1WY9Ol+cVNcSzozZxgn4K4hVlJT5+yo7BjCfcowWoOgbrI7JpLCYo7gB4GbVIh/liCtckxoFZ5kLof2zSO/AK8LwU9mTWIRjF5k4dLVMORj3t9UBLIWAWyOC2vJxlnotMTl1paxZuwpSnAi1aV4RrEOkpOA8ANaGtEgQPCEn5xBd6VwlBSt8fSDC/BguYXwoVTFsD7NLmkRi9jzNsCQsdj4xcdjMz/vQH3DRqGyTrFlE6Mgzqz45xN4kri7sOVvPCCFjoXIuidklKEjn2gbx4AA5DSU7eFEI+d72AQMHfQI+QllyU7e4cs4XcwTugJLkfaeE/aHThFtb+/mEfhFtf0pQIZtCs2Cg0c2FGdMG8hMiGqIsIQT8pulKGvylxQ4ao0baA4r3ILJn1DdKwZrAiLzqxpx65dBM53qd/CqyS7M2les/q9ZUcB7dMkPoARNH+WaC8FkQq0EX487//r2XxHd9UxyXnOkJsWNjeTkO5D6cmJUm4ACCuHQFTfUH/oATmoM+3TnVqfEMSK8ow2gejjTeOFZSDuDWa16Hoozxk8IO81KblMdeZti19Z/pAgf3gIXGoTWkrDS9FYM/FtflQmcApucRTEOGAZNzNQ7/0XHrogUtuKywBPJTuJSm9/IbU8JCtMYXS2aAWLrCVV/ok7cF1SfnKICwaHmMY6gWONc1jAqHvoqrygQtMbNIeD7uDXknn4kMrCiMTWnaoC5C3ktGnRD4g84DPS7SpzALcfubXproDueuHPIXGpd09fLeQLQuYYWCDOgZqkQaRW4OMU8Uv3MpTZcHouAXeloZ5FcaXZe5ZI1kEEZMJJGkgzoLPFuXbVJrqrCzqtWwnN05ct7WAz6X6nfAw6kJUSeIf/rQ3KYGS1L97lFu0wp0fK00xkmuu7iiMX+gTqUaRJe4ERztVI8QQ4a7NnozAyS2M5RFe3A1GaqzRJAzBhilUGKBL09xIX9wNTOs5gqZu2nd/phQiyvMwmRnS0I1Neoxynsep3NQ23mOTfGN4LjWosQOyClVKYyvuq/vqFW7KGUA9uo9w1jk24AcL5Ja1TsiKpzDAKr2JhCnalm0ji9r2rZTPa58QMPooB2tJ9xCsPMesTiSKJWnknxWvp2Cc+n4Dt0v3BuCjnSIlRZ41ZL2nK1O51ig9QA5x4PCAzBq1TmoUhQdnSHu05MRJCaTwWyA2j2hIEQNSuVMNTKo4pDH3iEJKlq3sg5IGKJlD04fj8xtDxSzJ9NpOJypko6ymA7LV32jTCIGjNxz6GRkMRo2uQQ9nkQtyxUzAoXZS4Y7U1GsNcVO1oQtUNnx/n5nbXUTw089hYbUpolx5ND6qvdDio0XRLtvjqrlJyN3NehwMC1MR7AeVPIWhOwUsNxNJp5NmRLkLXZS81nXrom6ovVeWKbhdIpD5c274mKmeuQ6RaDutRGB6f213EmskKwBjdKo0MJAL9F5v4N294mlz02LNqyDvseq1ENZ5GHiCwhn5PskNK22QS3O0tPTE9PdZ+MNhAbtTBo0dxc4cHOfk//na2zgv215b+7gsviU1dGiXBz0WRRIdvMJnqsWSTNYAVsY0t2V8qXQtaIKrB6BMt6ZEtF1dUV3SzDCkG23MvDg8fNgz8iQuelzx1tvaskRwH9FrdzEybn30Iol0StyDx+b4Y3TR4LJ/jJQgHF9gTD1P9NvMC9BlE1WrKHSdqssBlk71bq9mYnQq4lqiQqHAWkRHAFnOps42P//HP2f7+2+k0EF0I4XcEUXb36dtOJxxJcetChv0lgfmhRqwZpUJT525KYciIwfKSNzKM8o/pc82Uzm5oksee2AtoDeqw30bHGRYZhiDH+JL9GfGF7l/4nUkJYogWwBjwFpR1+2GwKTHDsBOD1SDPCj5VUgbv4Cj/u3BqmhwW3aDgZTQQKE96MQLgJjB9DanGJeJFWVNLTRZNvsQccl3QHiE1FGTcU54HtHfKjMhHrQLAwyL1H9RbQE4Ku3NAsXk+ygTysEGMsG79i2lHq6/jMctgNq6qMx+Bxuv2u9/+quN/Zb9bcV3cnUrG8bhTFPh4sYmPZuYhVB+yX0BpIORWKmDE/CGeOzShr5JuX+w6mIJTUIZ7ICW0xi3H9tFJAd3L4meclpy785Alfnf1rffvccsY5WOit4dRoelZdVeC9ZoCF4iJWQVdSjkvSArKMdbzHYSvihHYaq2nKoVBTODBAo+I8uRHrUbDb2dknQrUldnYCxmNese8ajX3oTXHba29G57POwfX1ixiHwwwE0DtZTB8kxeRQgEajx82LtQix8CULn8plimQxQg70gSP+cpV2gElNIyur/mEu5twJVeQdJQPLgx70Zivl53OmZ5IvFu66oy3VBRIw7GgCHLC1xxfbHXQHv3TgzQhyfmWer5RSyFbi0njG6qsPQNDIWW3CYVa9vttT4wZCNHKG2QlYw/VeSleoXZ55YfgquFlLA++GqauqamurgqqQKeJP9/2jd/XBNlN7OkAu2+hlanboFssDrmaNhAI0WJH5G378vPhLZmntYqB8TMcS5m0JXNYkfkOqZKi5IRoqTCCDqoEYL5nz9cLmC0sqG19OI0475Ne2U3pYgECnf6zHIyl1vw3kaDB7FiUgis9NJLbL7Ec0ETH+07QizuK/RKyrnJYKqmRawUilwV8TeFFbVvUd++PXsDCnm5sLJEuGAe4X5Fb2j1wXY8aNh8A1DQVqZB543R0Hz5JeacLqxIxJjlc/sbyrN+ef4dyxLu8mEzLHM27l+A0Tob94BYar2v6GW9l0pSeAEq35qnhD6HO3MB80JbTlh0l9szjbbM9Vo+FMbcxDUSpf3q9bv3FKFPdTR9FmX6chnzhb4uXTpFC9zgp+t8L1ZGTjWtrZMXEsgSiQTlQHQFf3pWb50uUUAWR9vIrDQC4FuPd5OFx7pdcFjjmy4MdoSgvY8c52vBfUfSJC26vqZF7Wzfb8WpoI2dxYLARCHIIsuN+RX0JqNvIydZdoJwLjhGvYMwAx5Ptgclk7nwfbUd6PAIodthlkZZqh5iIEFfgp8D70aHGGJz4SoHjOd1xjKcfC0F+DUdSdJbY3LbxgQr02gDuenbjlFKGEcXB/qIOhLrHNp1KC9zH66JGHHDgo8H+BjECx8fXuDfowIE0oOdXqcymqm+6vcGR0TWByv8OH8iadq4Q2mVQBq4+Q5cJT+5U9osuZ+QdeCkJpIBzg+AjZa4PQz/5PwKiIfXChRc2uo5qgZc9FEaFVmil3ZH4hbB6+zxYrjEL+NLO99ruS5EEuYy5u/Hw+5prxKMQoCdfL5wtcCV0I5ARV0UvkDM8ZBDnlJjnvYoRvQ2/BDSHq4bMu7P8GhqIpMpfQ9TNoB95vwGjG+/iNpV2Kt5MIDOnH/3dvCN9FGzSg5XJVOx9fSNnBmGfDDLEu5A2pOfpLKqtrnGH8JJIp/zFGVWAIts4EM8q2ZwJ8XYFG4toh0W2DhfvFOn2L5+/aZMIKePFIWcDiSJnPYkaZweK0I4OVSbluCAyfNvpoEEqaQi/PPgeHI8mfbw8jdo7JnGb0HLGFpWYovTyaPJdKpb7NWtX9Vct+5PT92HkxK8Z7I/R/Unerx32NsI7T3KXy2Hsf3RdHLonuLlT//4TLP/7zrsFcyGUnxPpbBFCwMowJmDe1qNO6D7mVZdR7kXiawBGO6wjIxHwlFBCxhByKm/EONLOmM6J6NwPu6CxHg6woWptMIsSZjq8dHJkUshf/qM7AHosOyg4Yuqh0bDre58ltqjkpzXkudp+inO2tCnTECa/Q7kIMZQTk5JmHVWr0B3v4KHmPF1TZKyTbJN6vCtukSFbq6roRuY/mlveFEzsWkhiujCkyj0bzClnEUwsDQZtVYw1tsWxpz8USsAuxazPuPwElhthSmQW0YyNWTLLuWHQvuBdVxFb996eNs6eFryo4kiSuNJIh5o1o64iztEw8Pomp3mmaV5BumgmkG6MZ1/1VJrfXjYqg6n9WDAB+7hw9atSniGGer81T50moQ++DabATubwLWerpA2btmTydMV0SEpqFV2i5nITw5wlhoLKEXQPQMxI9MdtGTJLVM8kIWNnrJeQVYVQiSz1Oe3ZLCFvj4T9EdF7jpiaZAbEKubrjVU+du3Tenk0IcK4SeEyRxHLSWL5ZlZxCkaswQW2xEu1jENysomWkd74gldXfdHrV6LXQ+AWn8H1H0D9ytkw57Vx9tB5VYzgprL8bF+1HV5MsfjNzej1hE7QhKnHtF0w0B6Q9q6xMbTn//279g7OpfhhvXDaLTEKmud8tcBdVJcfl94LLQhWVlZ1YMU9dDD3/xepgvKg2QaJXubDyyU07IHvcFgMFlniuqCylHVl76eet1AYpoKTwAwPcBsaUqUZvlBg8e1TP9tGd2VkxX1MxMwcUd0JyK9wrRyOjpxFfNoiH/ozAQdU1jhot2uddKAYW1zoE4Gx09ZN5SPK5/UXUEQAQd1wJudQEzrja7k7e/UspWNmPLgljPabfwyvAZ6Zj0GxMxWr4BksYiBiJfijLJj36LbUoj1Dcn8j/DoSL0PmRqPKGI9QJLmpdtVSx3UJM2CzAiD2X2NFNprqwEdaLAkw9q3tXMbe3vP3mFqN1qbk9C96TBLHnqgxeZ4co+tKscLHvVP+i77V9JD5OBW7zV8AS5JV+HjuAdYKLV/zCpkeLv3YO7GbN5nqyLBf4AUzBSmeqpJxFbV0zS6wQBRybBZ4rmiNtxJ3x0cPWZrfKe4S7/JmQlxyDQ34dkfBIvJNX8MuDhca+KHmKylepG31c6k7KjgUEFXCJF9VhblAWYVlqEgkfG4S2uFXtRpzxWzjjKMOw8eOUfudNpuWqjpdFp+vKeNpfw4xqPaAq/Lp9rr6y5IeTe8GiKLDY7gvRrH8TGMPPZmCbhEFhIwWynCvmX6uc8n8HdV7WRQJbLbvWkYpkiJmkmQVSsNkOLVnmLm+a6dedqB2+DUnKutskjl3YgFRbliiy4T60ueYLKoh+FEe56JkeEFrjfDVMlAZGkMjjA9TXywKrdHHxC3o3EB+mUYznzxFTw1ZS0k9DmMXLCjpkt4kHTRuMezDVYCuj3vAPnaLijBJpPUUJwM3za9t10eX5YaFcCQPBugSRpFaE0NNDjdqg6O1tImesNWRM4FrNLbHJBqUp0kJe5qKFK+6PkVrzQI/b42FiKvKiQpb/LxpF7qi+a+SnsFUZZuQ3pjg81Yp/MXdiJAxrpIXw0L4AymhzTIetvtI70LdI7xHeBvRK0zF84l8L3CbsPo9ZJvb7lx+WufNRPCxka1cUt+nwhKmikVlkIL7XxMIQN7NjHIJz48UaajrFElzyt56RwFw5dILolJ9DIyzs4Z5UngqRbc3scDBMjeI/oLo0uSESjjRsHwIRkZZnv0dOWGDvilQWohUVtYTiP52kvAf3Fds4WzwFIsbYz3uGJROt6th/Q2vDKFLxa2544M0KpGbd8XmrwBKCCb0bt+wOqD/j5Yi9ufV7blKx39gNE/dVCZksFrvd3/VB4e9sOj8R9/yDwRu2EFnO77GZ3eNRMHzOnRYQdDAbZ0pQ8HvQ7L54/6t46AMpz3fJKYDe9rbUxDnZk1NrRVCR94NIIUBiaXARmaRIYj41t50OHTj5Ny5l7ITCoJQhm17R1SaCu94Y72PPRBbo6MFx+sIeNsIUDxL8E+vgxwQeQMEs9Ht5vHizC+Zx+4NzQy3nl0ECsDbqGsBswEp9MNdIKjfrrBuFcPOsB/TclzfiZmdDrw8LnFvpKHKXFWH0I6BQp2lf/p7xPrnrOgMgCjAR4PurblzVF7IwR95AIW8TUdDlFL+FUlnwM5+yYClo88/z7DIWxRasHIePvyyzMjP+U/OOxtHpQmxTPHAU0JWDGNN1gTgjbnsdwQeDy+y7EUVqwyp+bhB46HmXAHu5EzN/WBbH1HU5KdILYb0PRCjqSKny3T7g+2TDtfQTFtWpJzmixKCuvX7Y/OU2zuEFjuV+xQHrha7+wVLCASnFzjWn/3Am7rgoLQxfM4jEj034mEsTxM1KEjJnTahNCClqh8IU+d0BtsswsbqC/xmwJO3gE+ops6fONih0MU3EfTe9OYtSAGTASq2glyiHEvnsuLJDDzrTfLuauWfbWBoiT+36IfbdKxJkzBwxwIOfAzWZKEDtjKwTYfONyaPa/VnvSjjO20UTYEjCcJZso/xVHgERKsWMVURS6VeVyuwNW5h5hXlVEOqFIKMxeZWAow2B2AG2K08obJBm2WMVkw5T7AzakfpgBq5sUcEy8mdDfB3IW29eRAzuoORKhTMnKBnikL0pQ7v7lY0yvv5rZi5/6aTmnU50KVeMKaM4iByEORrdABD12wFDKff44y1Xh5+FDVR6QtSvERuxsctu9H8Gc+pciDrvfvQexSOEl0vvPBAovNngX697jD+tYjeaZ4JA92KvQuRBqHyf3G9gZrNOCsdJ2we+i60mmvLYuu6ovR9Cm5c0M9sTt6qSECZj2wABU96yRHRYEJ9P0DsFq5ez9sPIMh4kf3QAKd7SpLpfGG89DyyNe7l8C8aHToQ2W7SOUmEd3ciZr/GxoU2lkYgr8vj0n20sPF+hv5cRk+AW9CmnD0np29vmP5KB6croltVQk5l9zoRObDPwdaoZqt95YOWjIo6TKUxZHJV5Bnw4/7AxAO/eMe6/V6ssYeIBBcdHAWMQV57YDKrqKCOqK6CwsASUfyQE4/LPXzuETzWNjF5aqS8+d1SHOi72WlQIvJMstDdqxKRXf0aHwqs7Z9nWBgNdtmbGCiC1Iu1nLGf7FYMx43kUuGWVZqGce4fBfjw4v2fRdMrbg+RmRG1+27B6pOPNakjirmVYMomYN/lp7JCepVGAcCq7QgRsqqABAzVGv52VqGKkYLqcIwG54iLVj9BrzyQKa/yNfBpx9Tz90q4T/DBcnPNm8T4v8m82CWGG917yIktUaYCbYNol4fWTvrfmL6t7KUrBlG5Nv6HVnbj8j8Xt5a6ZzwVmLStb5U0zfnR59LT4qgvsEEJgfkM57GHJYELJYtjOVmM7uhkvuCpaGLTMwTVik/u3fnaegaQ1Nd+9xULmrbq0mqsrWf/imgTCuCcBcv6tPj21aaVgtNy5BhIvVmiJNUGRKyik4eLmousWaUAlQzEaDV732EefmzLV2QkNuekVwuRE/5yDJfvz0enlxshIvV066HdzkKL2Td+sTESi8U2E5GY/K1Lgob+YOANhuZvBJOe+MlMupYLQe61rO9htkzBxMOvY+0/hJvyWIL3mbguBCAl2i0xDfavX4PHgjLglBWVOVYacwPVVlOncl+D06R5QQpzKR882MMRFKtNXmiQlwpxt4IdI75p3KwX3m+PFFmO2EWpGAhY3lxWIGC6P/d6/PS+aNdkH62xBo0KAaHe7+CCKaaBxXj9pCSsTqsqlsxm5Xs3P5nqthz+vUI8+e//C/9Hp6Mp6Lhd/C3rGJRGV0fswsfHq2Nr6/Hdzgo4gT0+ZY+qDpGFbz+WQj5b7WPge4DfzziPmhQ43lG5h7oCrSTdnGUdlk/WT0IGn+TLSY4g8HhUY6Bd2Lh6aoBwqH6caM7gk15haKtGunMFzzFMl7bmFYVGakiGH97A7xILFaj/KbjHOpX0IQ8IFlJDOPUW+DL8iMV6N3+CcClPxo+ZqFqNeqhNE88tvBQWqQxam3zp39s/zpLkdc/qRuyRzieRyhJ+j0a39FRLzdjcaNKf3hvZ6xKX2+Lqgo7qU9VhaWCwSP1UyswLPXa0FRfkJXyU5n8aYH7GU9fh7BmZvn47r1MpomtT2vfIWDpMUZbQccZFNk1Ci33VenY8Oaedo731eNLd0L+RebRndDX1O05HfPAQlaA/Ds/l0ely4HFu764r8bbxtTro/+tiFkm63pjmtkBkznjO5gNcTGyDXP5jJ1NkTgCVsbbYU/zUXkTE9OGmpQ2/hxPltRGuaFA8P6+skn39y0GeMHi8rG3EDEVMdnRBZTlVDpqo64UFLjBEqRU36IcclE/W7SrHYWbuMp4etSgaqp7trgBDgZ8ssPebT8v8YI7NUVRmM3iYmFj3RqJ2Ffv33xtrpWZPPsh+/Qj42A8CkeANiiVfdzBEhuUa5rIMeUhxW2jykMdOw6qsVrMTiNsqpKpRpqf89w8UhzZhoE2dA6ctiaaFcs/B5bhMf5kggncL2eBCbpf5PVPWAKCU6qCnaisUcRsJrYG311XZyVfl4N+Yz/9Q9XN/umfd1JMOxoIS/r9L4kbGoO5LteLbeSTLUba0gY3tgHQS+3hbgfTFIRXZ4iwOO35dyBg5SGau3YKFjrwTuRRPWJVEnyLcJPGgjfbXEN9aqtZ1n8IJ8pwQSvQNAxVkk/uBU2xwLN/Y8vwrzy4UqvTBRMEK6HDUEFoOx/bVb1sfaRTt8aoY1rqSnZgOXMezIRZ7rTD9L/j5qJ8wMkX7Q2AfL6YuJwFQwboGemspQJkqe5atRwZajw6cYTs2iHh2lGSsCNlT0fRYkdSUqfg3I7SRXS8ukCDCWskSwqbRTHaRj4d/5Eq1BbO2q9aprZqtf/atWrLu5KqVq2uSltev4utmAQqL5ZPLqZeI5z4BR4ZC5wsBkvcuSliUXuNwY2iIR1jRNcDy8+JmGz8/X2qai3B1wS8lWQOHhQ2y9XP1OjGiqrKo7vPuNg9eiYSV/0W5P5LOit6U9JIUXpZnHVU8VN4VjqJX2osuXXIzFLBmU5RjaCtAYxnOHqKEJU4Hx5asgCDWdQs0Z8Usyw+mJS6pkJLAHmiWZjclo72MS7oFXzekYb9hoUpHfwvwc5LG+jBkD1dPpG7CGWFjc9HAMEAtykTJkhtDrrcvotQMQOMFILOm1EyGkHt/fLfEKAaz/Ge/e7F29++eFutpz8F6WRjarqJf4qDxM88/F0frOlOdfRxm9kRvssJGJpeKZV7m2QJxSvBmqdtTN+bxGv135PQuRRFDV31W2ZegElvuAt2JSZlqw3jNHqsVEr3sqHwjou147CCSLV+3Nq55RwoUECYiJqztPmYeXHaUtVISeujMtIsCAQmIBSDw3IVzQNLLVX7PdVjQFREiIqY9CNiqsMIX1+wo167lkErcWjJf6Sbkeww7cSaeFiIyOhZ9D9UuA2mrq7u3oScb99RlbJ12JVKTRoCjF+Rls+zwJnbcplpeqNHJw/1xGiiozX6yzkI3hWnIwq+kkBNOhAVU1HUUWlq6jH1JfEpe1flEvCAn9iUv4TBcBGktixramMCdf5ZeGULxIF6oLPc+k0xUwqtC9emw1Sjcb0cS3kzRAkMTVVjzRJUrxlZf10UEM86+Q9Eu/nJkWtR+Y3fUhNjS/0j+KxcL7paPgNE1ExQYl33B/r7Le4QXW/J/TLmaRolw4ODGZBsNrGccFEaSXlQB2BwCJ6I5ADPpCTpAcYIUPAebHb5jdL3XUBUdt3lC/fhEUpQSrCvFauCuTpzkN04yw7rhY+OjyvSFR5XSRXkEP5qdAKS2aUy+1mCWcwcD8pr+u1XC/vp5tWiSvhkrcZ5FvsJnwrztP0vyQBlmmc8S+cjE2fVycfZkGCAzewFkAGWeG46PbA7r/wi9pCKkTxj6S4NjhcTo840qD0qHAOIhlv6wYk1uSj5pMpB5dKITrWKzDkV2hoTARXy3uh2g7CLP9Ymi03JR9DtLr7+lJgEeKQ/eETr1h+ucKludwmVyyovo/IIX5+/QBsV3I+4/Pzd++eA1FK1mNIvEGAxnaJMZqFn6MduqgU0U/aEPezVihvKA9jYkyo7YyEZ+NV6S2SAYk1VAaYgRkXMOBcPY9792Ouedi+++N6C5SlW43sUGRhZxS7kjznUa88t1hWRXPCFhcd1IrMhQgA2CS/qbG8qXK1NvjH96ASqZxvVc2meF1vqWleJUtHKhcLTLrVRZFlu41mOjFoFFBS+uZWBU14rnqsgfJfoXzMSAZ7GANtsAtZvWPqpHjrfEcv0/NCyyj9xQVKmbis1C6Q1oUSyRgqCZlGzWXI0kv79dWlBDjQwGhFK3oYlWpMbSi1Aayn5i4Hv/V9J+1S6", "estudio.html": "eNq1PNtu20iW7/4KRkmHZEJRN8uxJVMZx3EuA6eTidPTAxhGUiRLEmOSpSEpWbIiYL5gXxbY3bdFP+wC+7DAAvswb/sw+ZP+gv2EPaeqSBYl2XF3zzbQLbLq1KlT535O0X14z2detphQbZxF4WDnEH+0kMQjp0bT2uBwTIk/OIxoRjRvTJKUZk5tmg3r+zUA5sMxiahTmwX0asKSrKZ5LM5oDGBXgZ+NHZ/OAo/W+YsVxEEWkLCeeiSkTgtxZEEW0sFJmk39gGmd54cNMbJzmGYL/O0ljGXLet0d9e632q1Oa79fr09ITEN4d1t+exfewyCmvftt0vY7e/CazbPefbpPDyiFt2gKbwfkwCcE3ojn9e533a47bMIbuwQswwN/z13tPFq6bF5Pg+sgHvVclvg0qcPICnliucxfLCOSjIK41+yPaTAaZ71Ws/ld3yXe5Shh09jvzUhiIKVm32MhS+Q7UGP2h8CWXmt3MtfSRZrRqD4NrDqZTEJaFwPWGR0xqv3w2kpJnNZTmgTDPpvRZBiyq9448H0ar3buZ2yynLAU+MjiHnFTFk4z2odRIGYy74d0mImnRFIIj36QTkKy6A1DOu+PyKS3D4MkDEZxPYCd054HAqNJ/7oexD6d97p9hKxfJQCK/1nt2CkdLStoNk7NRWL2Bd96LTwoCwNfE7MooHyynhA/mKaCtgnxfeR3ZyJIa0/mYjvNnWYZiy3bzeKlsluWAHsmJAGSK1wGKRe7N9d2eqJsBM9aC3YREgniMTAaME2TFFBNWMA5cTUGxtRhG4/2YlbyQBJlswpJ99tee9ghG0KvrOkB+4gbUn/JAG2QLXp2p5vv69MhmYYZLpgskb+9Fjz7finqhIYkC2aUDw/WmXJ3EWyh0fc1O6LxtBBwzGLa31QyoVFNoWytbZr/G3VgD56jIBbOotc6wLmIzOvS2vaas3FpEWSaMU68zSY0XjuBGzLvEmZxUCOW+BVyqML05V5oyBmdZ3VuFj00o00b5gA+9VhCOGcEnyTx+6hXSHH1gHgmhU98SaGlt6hgQXxvjEeuHEEMbdHAfFEakTBcblgH8I1kPS5G8CR0PrlBj9hlISYVBw4jxfWrXB5NQOMyICna4pPERI/bGndLXeQxWu+QJZGwY9Bp+iejDjOm4n0qjoa7Ij9IqMfRA0HTKN7mvXLHhhojpHqwN7sCCkkcROlywwmiZKp+rv95mmbBcFGXMUxizlHkCvSb7U5xVbmatNfs4Jsuao0oextdEOq2CpKPi4H7w+EQUAXxkG0RoYgmELS4ze8hjZtKhRoBQZMKSedS3O3juXPTbdld2CXNyIi6xB/RJeLrorQkAFqzIrknTS65kI5o7C9vckV8Q6lmT9acxW533VmoZigzB8/71b6qMPe181d8aMGM4iw2+DfND2YVX1uedEPGYryyoqLAW+xAJilAjNZc5QiCZe7oSp5v8VYd5QRBzCUoXel98B3bJBHEkBOCK6tQt4Wq7aa1qU45yzqrHY/EM5Ku+/TfRdQPiGaUyrK3CwcxlzsaT45yzqCG7eeuZ79IiPYxv9A2EgzV9A6qUu3wFcJEEKnQJaF12zdQI4k8p4KvVeLT3GRNEbTcn8oNOk0BnTEWpsuqayvUuz6X0VCrOqr8UN0tuipPtWaS6tla3WblUFVFx8VSaSvw1YOqFtneA4ssT7DLT5CnQgft1c5q57Ahk/7D1EuCSaZhaeLUgghLi4hMaoOlfElrvWUtGyeU1nq1cZZN0l6j4fmx/Tn1aRjMEjumWSOeRA0O9Lum3dpr2s2GOw1CX4zZEfOnIYUVNUugagC/WJw2fhlKOicRpPFp43MaNWqrFRyCEw+naIjyCWsHeAMj1gLfqQGjofLRND7ghSRNnRroY41PYiGV8mkAEHLUfJKR+gwW0nktX8Di2uADqNk0IYcNATdYh4csGRg1eBfQa5IWUFtRi5COwDRl6Y0YYfNFbXCUeEEY3rzvOACYMCPqlocNOO3GqYERWyd8X3AjRKpqxQZyFgy2Nvj5X/6inU6vtZ//6X9KQhQUmAht42OIlS2vNmt52bn9IAg4B9sNWAKQ8kkzwO+Zt/ASVsEP+FSPwrKj/PHGLfyERJBNeqQ2eA6PX3/C5xuhwVPEAPkOfzQjgdo6JYmWCT0oyRI8vZWzHomi7Yz933/9h3/Tjr/+BHGE/GLuek6taTVrgxdYn9Ht5wCY1j6HOoEKK/TJjWAHHOqU+EwLrv9s38J2AK4r0BDS7BuxdrpWa7826DR2bwRpWvtN1PMkcG+mbgjVYW1wTFQL/DbjIe3ezveTOTo2EOetPM9xCAFU9st/cj+DEZvTNyIx8ATcHQ3Zz3/593W4MgbUNO5/gdi7VIKVfK9bSa9kf6XMMTdz2y35bzU89Xk3ZgzivOo1tabWwpRLYt7fLxxHeVwejTbHMdBujoqgVfphHOPB8waPVDhoHonXVB/HE5ZVnPMRhNI6DJJN2+f2l4F7OAYjSRSADaRXUPeAubPw63+OWLzFMXP5XdIQbenPUxrSjG2HGQN5YNr/+FftmEwqUaN0zvlPJfiKKAkHFoFXe6SRVPvw6v3JiTZMWKTpPBrq/Xx++TZxg+wYRJkAo1YqUB5fPTnZqIBCnFWwvDz98OIUFBiy4G0oQj6VNkqw6vLn74+O395hvQJXRfCesegkngUJi8HUsq1IaDmfNtYWCGw7cNQ00x44qTPwmTflMyCnZHEGsvIylhip2ZdQL16fnpw5S/DmvXNdWOvHK+rao9DVrXwAdRRMojooAkBwnU9cWDz0A5oJj/1yUAZ5HMYoL0cxpsMQEUFdDo4DpIHAwEcCkVyMrnJKv399fOIs12ns6S9Pn2kQsmnCNBjUDIhoHtMea79/d/LS3H4GsUiOaAYss7SXzGeZpX3gWdrnVFkJjsceunNY9eLZn7DVrNFc7TXjhxhSSUv7IU4oCS3tTQCJGYPFO5pcbrsheAhY/Ax/KQRzMWLezElBXhooG5XAkEdg6GY2cz8D5NtnvweojIdwpmz7EXQqoSl40I5vp1kIoGcfTjU5+vW/Y63zHOqZSCEjZH7rY7cpSTh9+zxna7f5XQWq/bHVVaCi6SKHbHW/4zQoGtDTRSqogYpr4ynoAOAqVaGnvyETArFT42Mwp+oEEA3eXMPeDUESqrqBa2FAwwENjgVxwuNHM5B3CfXhNcEDzgKfso+QfQQxSexosgtL/4hjQgf4ogJsBNbUgSRbBYP3v/0XhwDuf5xgDkTsSTwCgNcRBLIYD8AnNTGprwo7TLjYaeLE9Er4MPtH6r48fS/HjSWJ8YIiIGkvS6bUQgnRZEZBka8gej2bDoc04VMrMNscHVST2btgTsP32Bc03pBsjHW+Ie4+yhmrbZp9ddEZxDojiGOa/CjvR+DxFbdSFT2bZpMp+EkIpmfYknYE6WfvXz4rB0vE4NQpSHICBEvIo+OTsxdBGAXeh3JOoYQHWRi2acxb1A6esL9TeCwsYGxYBQuOx1BAGcVSH5xeSBGocGMplLxU4fAZvhtmn4/bZZaggPBTGM25uN8pME2ihEYK2Ls370/evAQsCUHPmROBvSwEB3cMFZHDV9norsXOuH7NORumBZXbrmkLc6f5hqCW6nYQICbYdpzRYxLBpgZkjqW0Goq0EF3LajebBe1eFnJUlRBnwAbWVt4BuGT+c6gmUXJcBDiMNfl7yCMyum3sbEJBXi27nW88SgKVtS/h9RUNJ6Dcraa1CynyvE07LfHQbu+2vVwyENQMXFwcYRgypprKG5qODYU9UITQl5RFNEsWRrsJxzctRexSqbBuIqGh3nqsTN6LZomN+RGmlvbcqXOzefe60c7noG4C1gs84ugloRykoHRMo0Ah9BW8ppMxMPkUhQOKNeT/wJE78M/urmXvqafG5aX6TmMF1fO88UzCNVxtwDCNbY+kWYVEGBP2ZEdkggaOhg7s2d238D/qvgBbbJsE0e3b7u8TAtu2VASwCBDsQFTSvCmIahqGVgTuRj4SjiJ1lisI8sy7VC0O38EMIMdWRp+xeQcGX4E+7VuYVgrNEdgwj0ZUZEYCodtncLDiACKj4uNlUmbktuljJsAnlYwLZvk4cug59SCoJe9ACQz913ZfwsBNGxxlYxRmw4Zu9gVZfAdlYw6DrBtOY84kjXcajEtsI2qC5FNnKRsFvfNS6tZ529q1OhdWKRHQJqtlNy2U/4WV9wzkorbvWx0blnUtUABc5g79fVhmdy37AJeKZZj6F40DsZY098CoAOK8vgsb88V7zSdN3BMwgiHxf8X6oo+gENuBPc47Vteqt3DtcNjtPnlioWxtILrVtYYkTGEt7yoo6+xd67wJy7pilTAdWLfH9xWLVueXF30gmtsAum9k8Ss6N07PmxfCNLCHHUPpuHBOz1sXfCgvJblN2LZ9et6+MO1oGmbBJFyceSQkifEKgmQQrWPtXIhhFesuJwFHK4jr7UevLPy33nkEyNC8K8u6F9KGlAQehp9cPBUhpIf6vgnyWkGxx3cOhsY9XGcut+zx2N7DnmahYS7YmmFii9+eTnzwi+DwkmD+I0sgoKIQ8b5hDl7jkp5Ek2xR2I67YaG4N+KB4nEGQYoazBksgRhmByl6aTN/ObvEIOWLMbCzS57BxpIAQMSAzREkF/QZhmRMcZDIvgvDkwWgcMthEzOAcCGo3oW5qKTfXPHSlYJmwDYjGRS+gbsAu/MmyJ9pDMw0XHO1wlwKojdwls0VPg/BEqhBrh2wDRo6rTa3acF9yVHPQUwjmh3z7rwS1P7IizJgsWmlORDP0raB9F85ItcjcyO1FxbkAEI0wuWpk3MLAVL72nzUsg8afCYjMSYE9pDNHuXBD7JbEyJggYQ45LqYxK4ZdWioDnBQsmYAnj1/zGEg+TaIKeA9lhrUfORbnr0oZ+XI9eMCZh1eHAmSDvBOwA0hPbxAhF1jShLHb0Diw1+H+PaoK16Ejr1L2GcRyoRMDZHrFAqoWgiWUymYCMazzGlaM6fZv03NHz68x+wpJOjPwXPa2CsB8xJJkFOqV3/22BnZJIPdXVDGtGSVB3qX9TOc5k2sp/JXTDQ6vVuWNTqogIWc7qAsAPvA0Pm9j27aInn88ObU+fRgyfnNk2Ijg6SUnTL8QOoMto5HhrnS4OHrT/FoGrJU+9tftQfL2Rag2df/SCAA0PTQTQYPlqBzAPQCsgHfaMP013/WcHCxbfC6Mhh9eozWbCClNi8T06efxL7K0EpUkOmnnq6D+ZN0EXtaIUo6hwosNYbmEiQghZI45IoEkFfSzBsbQ2sJ4hkzv6e/Ojl6rpfmDAXP5cojCAQClYM86qxWO+sbpWN2ZeCliRK5PRL7qcPbKec4ddFHjRoGIeWZTB9qWEOmuBobCnAT9IpnNvaYIN3mksMP+25CyeVKOnscMwVFuTwxwQB58sapLfumjo7XoHqpHiN5dJmN4M8RnsNAhGDTXK0pSCEjoalgxfrkWH5Q92mzk6tVVKfVbD6iHDf1Gzmy1XefhJ7CBmBL5lKENSiR2IzykRXMlLneGoAy0V9LCFciCIHx8BX92wMS28iU2bYMn0HhNoWsKzqGHaCw4WLv8+AC6FCYjuPo2LLSv3zJX8eBbvJIwUuNtYrlDNysD3lVWYmIrjQkYgeu5+1ZwLzROKZp2rO7XQu/aQzFW7MrOFfZWvbQdDzRnXcEnoAkeZUrGwvKpk9WEMvETmVmj4Lpr4etKOZeJgpie8HZXbikRd2JYlk3lYPgPrGgq44A0rJE6JclQh/VjnfAK95J16XygC/EBg0Ht0MajzKQqsRUHP8oB3mDE8UpcjPgLX/nXH9PgSRsPr2Cipck8PAmSPjvUQqqHuDT93TER85Ygl0Y7Dm9CLGdDg8vJfR76sqRY9FQwieWQHGNC0k49Yl8ElDPwL7FdiTmDa53LKUfP+gXgsjKEVOWZIZBLNd0BgYnXASHt+AnbPzo1Xx8cGB+d3BQX5t1K7Py/KXL8dDlqDuZSymEc4+vvHA4V20vDCZHfAajLceS89F1itaMB+4po7KPYOiip69jlqX6DoEZDA68k0cN/SOcXeNQLIZ9oDQ04JjougwBalbUQW39uDLX48A55blEL57mTz31iFASSKziaDL6Y9OEY0P3DG73DByMtPibnIma0OYrZDsOCUYXBfTmn1o69ySMOsnvaE4hPOU+TucMQ/Oj2Y+QRBgIjNlEBRgvLUgQpwK8COfi1mkjAKy7i9JZ8c68/lTnn7PoPR2v2PQ+HK/sLB3u7TbNCvKSDHQOOui6vklBabTruz/VD93iZn+A+cEx8YnGHaHmaNNYxHIbZ54z0GNghRbRa9g2trSQaLzbi4M+A4O19R5XgfUj4SaiwQzxCWIDTSZM3mYO9Mcc6pQTa5g94VdErg5VOq++QeinovcpM8H+zQFWsA1QbL9WOQpDWMo/ptBym4BEOzkhkFW4zsBVWJqx0SgUamC5Nt7ugiLYMzgZzyuqOWrlFDKxSSkVPaM3ZILO+lelrJFzlCRkAXD81yjDi/m0fAZL6pVv0jVHDx9G3L4AO9LCUxgxYpp8AIsCMWDp9/XHeXE94sV1nkKaMgrJjOsc6nNcfIGNLMM4jy3vApzEp0N+5RzkV8XKte+DpbfC69MA0s94JW4TP5n2ZwZ1Bk8SZaMKo4QIPwpf0Z3EOTOg7JGeJb5A27hH8qxLpDGI4eFD+XDPcYgpn6EE8enbaWbYHbNPbGzfZ4bJR1/HOGjzjbiQOBnkWzqkflF1Vx1SXa/jxBt+F1itVD3S67CYl+33MGjKwwpmXKXO+cUNWlVVpSuoUqbp2GAmSBJecmKvnMGVLb6Zz53elXljF+FerqxfvqxvUKa+XPE2WgzGXXS4d14+X5QcjZxBxHfhTsEBdsiCYFVsV+jH1Xpbuiz3rOrMM5IG3mbut+/uu8OhVWwnsjLlbwvEgNK0tnw6ycY/JkFGe6IYMYuofFVlk0hlr2zR6X/Lk542pLbou6/QylZFz5b7O0eXjUZduXPZVMX8w/91LcxDOJYRFPwkm0DRPSEjHnvLDqzjShWQyYIcZ46/PcpByBdB5naSFGrmzmC+LbxyTCa3Y2b668FMTKPrKTaC8ZMZPCAQXvcYOj+hbmGW8ncix7yV1+fii6uLm3iNhCwL+ZVRI1wPZau77ON9Y5/qLZDIkDA0Jiy7NaEJMBvNSfMwVOM3S1C7/OJumATNnM1Jz55bonE5hzrk1SO7+QQbSnniv96bygA8sxfw7/VjAO52RUNIbS5laz2i0g0IKs6JBVmycjIbkoIAsl8LuIHB6vtp5GLFKluBFv2WGG7NFFRZ8HaDkiSgGlWSMBX4hhQujxUyiysFeZvQ71Xft4pfDUJVcHMl9vGyZG2fPAPbUZLeKh3CS8TOvTukxf2tMCpd6NaViCfpkqn5LRtvJO9bNt6AWd94pwgjai+jvLKU7XF5X8qr1xKwiFk2jwQfwGdLU9wGg9/Oi7+AbKs3fko7RZjHrU2T3xhOl5GtRDQRlSJbxjSHX8TyYCQb93dt/PT/v4nOKWz1qweQYRdpzvVmvGk4xf26eDD4sSzwQ7kTIzdWz0THpHEMOYGz5ZYe1AlD/A/vTw09wE9NGvjZCa7w2VWMZYqje+LjOv5BCkxwsgzU8s2Qhh//XFMR05boJgn/2sDZ/oXB7S31u35Usv7VStGvOWUMcvzS7PzM4VfGGBae0zAjhqzdodRQS3h/01ffxH3ce8fg7Vu+Eagd9ob5tbLojhpqf1hPgtFH3m/+nPKKm/8CL4vOcBkQoDotLqTfusgce0ZCKGlFD5sk3jiYsfTLl6VM2tY6wBufulm/7fs7S/3qSv3EqvpJ1YUJu4rPGbZRJU9ySRepgV/cmWvgglugDVGQgnuBOIYlG/yKik02yoV5CtiiK2+KVrfAJXnya8Nj2WwRHXclOl7YKSTlBtBQaa0r9yaRg2Ysv2fW78aFjU79LzDroWqt+FI2TRD5+fDi8SftkP/9JBSxwzyxsOHQE7AQE/+6g09+At+ktsUI96NrtHMykd57Evlvo1UtK4db9xfY/8CNAfzUGUXVf0cgxqcGmDORWRiOcl5jFTJz/oBWbugz3fzyRc/w4gJJlvcnN8lw5qiS4cCgGEHsG5fOQKy9vGktbMNb+DIgy/1F3oBZaks3tyYSol6Qya386CvLyacZmSD38CYlyxntcv0qv3Df6IzSDAbWekuiNbcqTEzcMWEfyMRMqfgDn8MG/8uew4b4nyf8H2vroOg="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and not os.path.exists("/content/TripoSG"):
    sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    r = subprocess.run(f"{sys.executable} -m pip install -q diso", shell=True,
                       capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else "diso no compilo (TripoSG desactivado)")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
if CARGAR_TRIPOSG and os.path.exists("/content/TripoSG"):
    try:
        snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
        snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    except Exception as e:
        print("TripoSG no se pudo descargar:", e)
        CARGAR_TRIPOSG = False
if CARGAR_TEXTO_A_3D:
    try:
        snapshot_download("Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled", max_workers=16)
    except Exception as e:
        print("Texto a 3D no disponible:", e)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
if CARGAR_TRIPOSG:
    eng.load_triposg()
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()